# Intro to CometMirror
This notebook is all about running and configuring CometMirror. In this notebook, you'll learn how to:
1) Load up a pre-defined scenario and run a simulation.
2) Specify custom time-dependent trajectories for simulations (e.g. current ramps, impurity injections)
3) Save simulation results and Xarray or Pandas dataframes.
4) Generate random walks for a subset of the parameters
5) Configure batches of simulation runs using `MultiCases` and `CombinatorialCases`

Let's begin by loading CometMirror for the SPARC PRD and print out the initial `State` and `Params`.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
from pprint import pprint

import popsim.config as config
from popsim.scenarios.sparc_prd.comet_mirror import build_comet_mirror_config

# Ignore the xarray warning about stripping away units.
warnings.filterwarnings("ignore", message="The unit of the quantity is stripped when downcasting to ndarray")

# Initialize the simulator.
model, state, params = build_comet_mirror_config()

# Make a time base for all of our simulations.
time_base = config.make_time_base(t0=0.0, t1=5.0, dt=0.01)

pprint(state)
pprint(params)

## Simulating with State + Params

The `State` dataclass is vector of variables that are being simulated, while `Params` can be thought of as boundary conditions and/or assumptions. Letting $\mathbf{x}_t$ denote the state at time $t$ and $\mathbf{p}_t$ denote the params at time $t$, at every time step of the simulation, what essentially happens is something like this:
$$\mathbf{x}_{t+1} = f(\mathbf{x}_t, \mathbf{p}_t)$$


## The Magical Params Dataclass
The `Params` dataclass is where the magic happens, and where you get super-powers. Every element of it is configurable by you. Let's begin by defining a simulation time-base and defining a current ramp that starts 1 second into the simulation. While we're at it, why not define a auxiliary heating ramp rate, and also a tungsten impurity injection that occurs between (2.0, 2.1) seconds in the simulaiton?

In [ ]:
import dataclasses

import jax.numpy as jnp

from popsim.enums import Impurity
from popsim.simulators.comet_mirror.simulate import simulate

# Create
new_params = dataclasses.replace(params)

# Manually define a current-ramp where the key is the time in seconds and the value is the current in Amperes.
new_params.plasma_current = {0.0: 8.7e6, 1.0: 8.7e6, 5.0: 4.0e6}

# Manually define an auxiliary heating power ramp where the key is the time in seconds and the value is the power in MW.
new_params.P_aux_MW = {0.0: 11.1, 5.0: 7.0}

# Manually define a quick tungsten spike. Note that under the hood linear interpolation is happening, so we need this
# perhaps somewhat awkward definition.
new_params.fueling19[Impurity.Tungsten] = {
    0.0: 0.0,
    1.99: 0.0,  # Start ramping impurities.
    2.0: 0.1,  # Impurity injection.
    2.1: 0.1,  # Impurity injection holding.
    2.11: 0.0,  # Impurity drops back to 0.0.
    5.0: 0.0,  # Impurity holds at 0.0.
}

out = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)

## Visualizing Simulation Results

In [ ]:
import holoviews as hv

from popsim.visualize import visualize_time_series

hv.extension("matplotlib")

visualize_vars = [
    "aux.params.plasma_current",
    "aux.params.P_aux_MW",
    "aux.params.fueling19.Impurity.Tungsten",
    "state.stored_energy",
    "state.density_state.vol_avg_ion.Impurity.Tungsten",
    "state.hmode_state.hmode",
    "aux.Prad_imp_MW",
    "aux.P_rad_MW",
    "aux.tau_E",
]
visualize_time_series(out[visualize_vars])

## Saving Simulation Runs
The `simulate` function outputs the simulation data as a `xr.Dataset`, which you can save to disk as a `*.h5` or `*.nc` file (`*.nc` is just a special kind of `*.h5` file). Alternatively, you can convert the `xr.Dataset` to a `pd.DataFrame` and save it in whatever pandas format you'd like (e.g. `*.csv`, `*.parquet`, etc). The code below shows an example, where it saves to a temporary directory.

#### **Best Practice:** `xr.Dataset` is the preferred format. If you convert to pandas, it will be a bit of a headache handling profile variables and multi-simulation datasets.

#### **Allen's Soapbox:** Saving simulation runs as data is useful in certain cases, e.g. to enable interfacing with other codes and analysis suites, but if you want to share your simulation results, it would be better to just share a notebook + git commit, as that will allow others to reproduce your results and increases traceability.

In [ ]:
import os
import tempfile

with tempfile.TemporaryDirectory() as temp_dir:
    # Save the dataset to NetCDF format
    out.to_netcdf(os.path.join(temp_dir, "data.nc"))

    # Save the dataset to HDF5 format.
    # Note that we use the same method as the netCDF format
    # This is because netCDF files are also valid HDF5 files.
    out.to_netcdf(os.path.join(temp_dir, "data.h5"))

    # Convert the dataset to a pandas DataFrame
    df = out.to_dataframe()

    # Save the DataFrame as CSV format
    df.to_csv(os.path.join(temp_dir, "data.csv"))

## Directly Providing an Interpolated Function Instead
Well ain't that nifty?

But what if manually writing out times and stuff is a bit tedious? What if you want to have some other code that generates a sequence of times and values and you want to use that instead? Sure, why not. One go-to place is the `popsim.interp` module which we use below to specify a current ramp trajectory.

In [ ]:
from popsim.interp import interp

# Specify a ramp rate and create an array of plasma currents.
ramp_rate = -0.1e6
current_trajectory = 8.7e6 + ramp_rate * time_base

# Interpolate and apply the interpolated trajectory to the params struct.
new_params.plasma_current = interp(time_base, current_trajectory)

# Simulate the new trajectory.
out = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)
visualize_time_series(out[visualize_vars])

# Running Batches of Simulations

Okay, running one simulation is good, but running a gazillion is great!

Often times, we want to run a whole bunch of simulations with different settings. There are three ways to do this:
   1) Providing a list of params.
   2) `MultiCases`: Specify multiple simulation cases.
   3) `CombinatorialCases`: Automatically generate all possible combinations of simulation cases.

As the names suggest, the former helps you specify multiple simulation cases, while the latter helps you automatically generate all possible combinations of simulation cases. Let's start providing a list of params.

Imagine you want to compare a couple of cases:
   1) Baseline
   2) Impurity injection of tungsten
   3) Impurity spike of argon
   4) ICRF coupling efficiency drops suddenly

Well, we can run these different simulatoin cases, no problem. 

In [ ]:
tungsten_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}
argon_impurity_spike = {
    1.99: 0.0,
    2.0: 0.1,
    2.1: 0.1,
    2.11: 0.0,
}

icrf_drop = {
    1.99: 0.9,
    2.0: 0.5,
}


# Define baseline case as equivalent to the initial params (note the usage of replace here essentially is making a copy).
baseline_case = dataclasses.replace(params)

# Define the tungsten case.
tungsten_case = dataclasses.replace(baseline_case)
tungsten_case.fueling19[Impurity.Tungsten] = tungsten_impurity_spike

# Define the argon case.
argon_case = dataclasses.replace(baseline_case)
argon_case.fueling19[Impurity.Argon] = argon_impurity_spike

# Define the ICRF drop case.
icrf_drop_case = dataclasses.replace(baseline_case)
icrf_drop_case.fraction_of_external_power_coupled = icrf_drop

cases = [baseline_case, tungsten_case, argon_case, icrf_drop_case]

out = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=cases,
)

In [ ]:
import jax
from popsim.stochastic import generate_random_walks

diffusion_mags = {k: 0.15 for k in params.particle_confinement_scalar.keys()}
n_samps = 10
random_walks = generate_random_walks(
    jax.random.PRNGKey(42),
    n_samps,
    time_base,
    params.particle_confinement_scalar,
    diffusion_mags,
    return_interp=True,
)
pprint(random_walks)

You may recall from earlier that `particle_confinement_scalar` is a dictionary in the 

```python
      particle_confinement_scalar={<FuelSpecies.Tritium: -3>: 3.0,
                                    <FuelSpecies.Deuterium: -2>: 3.0,
                                    <Impurity.Helium: 2>: 10.0,
                                    <Impurity.Oxygen: 8>: 10.0,
                                    <Impurity.Tungsten: 74>: 10.0},
```
However, the output of the random walk generator is instead a list of `LinearInterpolation` things. But have no fear, we can still just wrap it in a `MultiCases` and it will work just fine.

Note that we can also use `MultiCases` with other variables, but we need to make sure that every `MultiCases` object has the same length or things will break. Just for the heck of it, let's also vary heating power across the random walks.

In [ ]:
new_params = dataclasses.replace(params)

# Apply the random walks to the params struct.
new_params.particle_confinement_scalar = MultiCases(cases=random_walks)

# Also vary heating across the cases. Note that we need to make sure all MultiCases have the same length.
heating_scales = jnp.linspace(0.8, 1.2, len(random_walks))
new_params.P_aux_MW = MultiCases(
    cases=[new_params.P_aux_MW * scale for scale in heating_scales]
)

out = simulate(
    model=model,
    time_base=time_base,
    initial_state=state,
    params=new_params,
)

# Running Batches of Simulations with CombinatorialCases

Okay, great! So `MultiCases` allows us to specify multiple different scenarios. But sometimes, we want to generate all possible combinations of scenarios. For example, perhaps we want to scan auxiliary heating rates for every random walk trajectory we saw above. This is where `CombinatorialCases` comes in.

When you specify `CombinatorialCases`, every list gets combined with every otehr one in the `params`. So in the example below, the number of simulations run will be the product of the number of elements in each list (3 * 2 * 4 = 24). Oh yeah, you can also specify time-dependent trajectories in both `MultiCases` and `CombinatorialCases`!
```python
params.var0 = CombinatorialCases([1, 2, 3])
params.var1 = CombinatorialCases([{0.0: 1.0, 1.0: 2.0}, {0.0: 1.0, 1.0: 3.0}]) # Specifying different time dependent trajectories
params.var2 = CombinatorialCases([500, 200, 100, 50])
```